# 00 · Setup

Run this notebook **once** before anything else. It creates the Unity Catalog
structure required by the entire pipeline.

| Resource | Type | Purpose |
|---|---|---|
| `asset_mgmt` | Catalog | Top-level namespace for the project |
| `asset_mgmt.bronze` | Schema | Raw append-only Parquet landing zone |
| `asset_mgmt.silver` | Schema | Cleaned, validated Delta tables |
| `asset_mgmt.gold` | Schema | Analytical tables consumed by Genie |
| `asset_mgmt.bronze.uploads` | Volume | Drop zone for incoming Parquet files |
| `asset_mgmt.bronze.prices` | Volume | Append-only Bronze store (partitioned by ingest_date) |

**Idempotent** — safe to re-run. All statements use `IF NOT EXISTS`.


In [ ]:
# ── Catalog ───────────────────────────────────────────────────────────────
spark.sql("CREATE CATALOG IF NOT EXISTS asset_mgmt")
print("✓ Catalog: asset_mgmt")

In [ ]:
# ── Schemas ───────────────────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS asset_mgmt.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS asset_mgmt.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS asset_mgmt.gold")
print("✓ Schemas: bronze, silver, gold")

In [ ]:
# ── Volumes ───────────────────────────────────────────────────────────────
spark.sql("CREATE VOLUME IF NOT EXISTS asset_mgmt.bronze.uploads")
spark.sql("CREATE VOLUME IF NOT EXISTS asset_mgmt.bronze.prices")
print("✓ Volumes: bronze.uploads, bronze.prices")

In [ ]:
# ── Verify everything exists ──────────────────────────────────────────────
print("=== Schemas in asset_mgmt ===")
spark.sql("SHOW SCHEMAS IN asset_mgmt").show()

print("=== Volumes in asset_mgmt.bronze ===")
spark.sql("SHOW VOLUMES IN asset_mgmt.bronze").show()

## Next steps

Once this notebook completes successfully:

1. Upload `ohlcv_seed.parquet` to `asset_mgmt.bronze.uploads` via the Catalog UI
2. Run `01_bronze_ingest` with `source_type = seed`
3. Delete seed file, upload `ohlcv_batch_2.parquet`, run `01_bronze_ingest` with `source_type = batch_2`
4. Run `02_silver_build`
5. Run `03_gold_build`
6. Schedule `00_live_pull` as a Databricks Job (cron `30 22 * * *`, timezone `Europe/Paris`)
